# Initialiser client Boto 3

In [18]:
import os
import io 
import boto3
import json
from pprint import pprint
import pandas as pd
from tqdm.notebook import tqdm  #  Barre Jupyter native (bleue)
# ou : from tqdm.autonotebook import tqdm  # auto console/notebook

import pyarrow.parquet as pq
import pyarrow.json as paj
import pyarrow as pa

from tqdm import tqdm
import time

endpoint = os.environ["S3_ENDPOINT_URL"]
bucket = os.environ["S3_BUCKET"]

s3_boto = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)


# Tools

In [27]:
def get_df_from_s3_parquet(s3_client, bucket, key) -> pd.DataFrame:
    try:
        # Télécharge bytes
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        data = obj["Body"].read()
        
        # Parquet → Table → DataFrame
        table = pq.read_table(io.BytesIO(data))
        return table.to_pandas()
        
    except Exception as e:
        print(f"❌ Erreur S3 : {bucket}/{key}: {e}")
        return None


def get_df_from_s3_jsonl(s3_client, bucket: str, key: str) -> pd.DataFrame:
    """
    Charge JSONL depuis S3 → DataFrame Pandas
    """
    try:
        # 1. Récupère fichier
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        
        # 2. Bytes → lignes texte
        lines = obj["Body"].read().decode("utf-8").splitlines()
        
        # 3. Parse JSONL (ligne par ligne)
        records = []
        for line in lines:
            line = line.strip()
            if line:
                records.append(json.loads(line))
        
        # 4. List dicts → DataFrame
        return pd.DataFrame(records)
        
    except Exception as e:
        print(f"❌ Erreur S3 JSONL {bucket}/{key}: {e}")
        return pd.DataFrame()  # DataFrame vide

def get_all_df_from_s3_jsonl(s3_client, bucket: str, prefix: str = "") -> pd.DataFrame:
    all_records = []
    all_errors = []
    # 1. Liste JSONL
    jsonl_keys = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.jsonl'):
                jsonl_keys.append(obj['Key'])
    
    if not jsonl_keys:
        print("⚠️ Aucun JSONL")
        return pd.DataFrame()
    
    print(f"📂 {len(jsonl_keys)} JSONL...")
    
    # 2. tqdm.notebook → barre Jupyter !
    pbar = tqdm(jsonl_keys, desc="JSONL", unit="fichiers", 
               bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")
    
    for key in pbar:
        try:
            response = s3_client.get_object(Bucket=bucket, Key=key)
            lines = response['Body'].read().decode('utf-8').splitlines()
            
            n_new = 0
            for line in lines:
                line = line.strip()
                if line:
                    all_records.append(json.loads(line))
                    n_new += 1
            
            pbar.set_postfix({"nouveau": f"{n_new:,}"})
            
        except Exception as e:
            all_errors.append( (f"❌ {key.split('/')[-1]}: {e}") )

    
    # 3. DataFrame
    df = pd.DataFrame(all_records)
    print(f"✓ {len(all_records):,} records | {df.shape}")

    if all_errors:
        print(f"⚠️ {len(all_errors)} erreurs:")
        for err in all_errors[:50]:  # top 50
            print(f"  • {err}")
        if len(all_errors) > 50:
            print(f"  ... et {len(all_errors)-50} autres")
    return df


# Load ROME Referential

In [29]:
df_referential_rome = get_all_df_from_s3_jsonl(s3_boto, bucket, "france_travail/bronze/rome/")
print("Total rows:", len(df_referential_rome))
df_referential_rome.head()


📂 5416 JSONL...


JSONL: 100%|██████████| 5416/5416 [01:41<00:00]


✓ 522,000 records | (522000, 45)
['❌ part-000001.jsonl: Unterminated string starting at: line 1 column 1217 (char 1216)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 755 (char 754)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 984 (char 983)', '❌ part-000003.jsonl: Unterminated string starting at: line 1 column 1126 (char 1125)', '❌ part-000006.jsonl: Unterminated string starting at: line 1 column 1326 (char 1325)', '❌ part-000003.jsonl: Unterminated string starting at: line 1 column 99 (char 98)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 4814 (char 4813)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 835 (char 834)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 1709 (char 1708)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 1311 (char 1310)', '❌ part-000001.jsonl: Unterminated string starting at: line 1 column 1412 (char 1411)', '❌ part-

,id,intitule,description,dateCreation,dateActualisation,lieuTravail,romeCode,romeLibelle,appellationlibelle,entreprise,...,permis,competences,deplacementCode,deplacementLibelle,qualitesProfessionnelles,experienceCommentaire,langues,complementExercice,code,libelle
0,203ZRBZ,Technicien maintenance (H/F),Poste : Technicien de maintenance (h/f) à SAIN...,2026-02-13T09:03:06.794Z,2026-02-13T09:03:06.794Z,"{'libelle': '27 - ST PIERRE LA GARENNE', 'lati...",A1101,Conducteur / Conductrice d'engins agricoles,Conducteur / Conductrice de machines à vendanger,"{'nom': 'GROUPE ACTUAL INTERIM', 'description'...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9078106,Conducteur spl citerne pulvé (H/F),Opportunité de Carrière : Conducteur Super Poi...,2026-02-13T08:44:51.000Z,2026-02-13T08:44:51.000Z,"{'libelle': '22 - Loudéac', 'latitude': 48.176...",A1101,Conducteur / Conductrice d'engins agricoles,Conducteur / Conductrice de pulvérisateur,{'description': 'Actual group est le 5&egrave;...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9076537,Chauffeur PL H/F Bègles,Rattaché(e) au Responsable d'agence ou à la ce...,2026-02-13T08:06:08.000Z,2026-02-13T08:06:08.000Z,"{'libelle': '33 - Bègles', 'latitude': 44.8081...",A1101,Conducteur / Conductrice d'engins agricoles,Conducteur / Conductrice de tracto-benne,{'description': 'Loxam est un Groupe mondial a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9076169,Tractoriste agricole (H/F),A la recherche d'un nouveau projet professionn...,2026-02-13T08:02:07.000Z,2026-02-13T08:02:07.000Z,"{'libelle': '49 - Doué-en-Anjou', 'latitude': ...",A1101,Conducteur / Conductrice d'engins agricoles,Tractoriste agricole,"{'nom': 'Temporis Doué en Anjou', 'description...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9060504,"""Conducteur d'engins agricoles F-H"" (H/F)","""""""Cuma d'une dizaine d’adhérents avec des act...",2026-02-13T03:00:37.000Z,2026-02-13T03:00:37.000Z,"{'libelle': '56 - Guer', 'latitude': 47.903715...",A1101,Conducteur / Conductrice d'engins agricoles,Conducteur / Conductrice d'engins d'exploitati...,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Data Set

In [22]:
key = f"france_travail/gold/datasets/rome_dataset.parquet"

df = get_df_from_s3_parquet(s3_boto, bucket, key)
display(df)


,id,text,romeCode
0,203ZRBZ,[TITRE] Technicien maintenance (H/F)\n[DESC] P...,A1101
1,9078106,[TITRE] Conducteur spl citerne pulvé (H/F)\n[D...,A1101
2,9076537,[TITRE] Chauffeur PL H/F Bègles\n[DESC] Rattac...,A1101
3,9076169,[TITRE] Tractoriste agricole (H/F)\n[DESC] A l...,A1101
4,9060504,"[TITRE] ""Conducteur d'engins agricoles F-H"" (H...",A1101
...,...,...,...
551245,5075589,[TITRE] gardien equipements scolaires non loge...,N4403
551246,3971310,[TITRE] chef fe secteur compétences H/F - EC19...,N4403
551247,2762776,[TITRE] Opérateur / Opératrice triage du résea...,N4403
551248,2613334,[TITRE] chef fe atelier H/F - EC22135\n[DESC] ...,N4403


In [25]:
# Group by and sort
df_by_rome_count = df.groupby("romeCode").size().reset_index(name="n_offres")

# Merge with Refential to have label
df_by_rome_count = df_by_rome_count.merge(
    df_referential_rome,
    left_on="romeCode",
    right_on="code",
    how="left"
)
# Drop duplicate key
df_by_rome_count = df_by_rome_count.drop(columns=["code"])

# Rename Keys
df_by_rome_count = df_by_rome_count[ ["romeCode", "libelle", "n_offres"] ]

# Sort
df_by_rome_count.sort_values("n_offres", ascending=False).head(50)

display(df_by_rome_count)


,romeCode,libelle,n_offres
0,A1101,Conducteur / Conductrice de tracteur enjambeur,611
1,A1102,Débardeur forestier / Débardeuse forestière,402
2,A1201,Homme / Femme de pied,153
3,A1202,Agent / Agente technique de l'environnement (ATE),134
4,A1203,Agent / Agente technique (espaces verts),2707
...,...,...,...
965,N4301,Agent / Agente de conduite du réseau ferré,237
966,N4302,Agent commercial / Agente commerciale et de co...,94
967,N4401,Agent / Agente de circulation du réseau ferrov...,134
968,N4402,Technicien / Technicienne d'exploitation et de...,40


In [43]:
import os
import pandas as pd

import io

#key = f"france_travail/gold/datasets/rome_dataset.parquet"key = 
key="france_travail/france_travail/bronze/rome/rome_metiers.jsonl"


print(key)

# Télécharge via Boto
obj = s3_boto.get_object(Bucket=bucket, Key=key)
data = obj["Body"].read()

# Lecture Parquet avec PyArrow
table = pq.read_table(io.BytesIO(data))
df = table.to_pandas()
print(df.head())



# Lecture direct ( update fsspec + s3fs package in requirements)
path = f"s3://{bucket}/france_travail/gold/datasets/rome_dataset.parquet"
#paht= f"s3://{bucket}/france_travail/france_travail/bronze/rome/rome_metiers.jsonl"
df = pd.read_parquet(
    path,
    storage_options={
        "key": os.environ["S3_ACCESS_KEY"],
        "secret": os.environ["S3_SECRET_KEY"],
        "client_kwargs": {"endpoint_url": endpoint},
    },
    
)

df.head()



,id,text,romeCode
0,203ZRBZ,[TITRE] Technicien maintenance (H/F)\n[DESC] P...,A1101
1,9078106,[TITRE] Conducteur spl citerne pulvé (H/F)\n[D...,A1101
2,9076537,[TITRE] Chauffeur PL H/F Bègles\n[DESC] Rattac...,A1101
3,9076169,[TITRE] Tractoriste agricole (H/F)\n[DESC] A l...,A1101
4,9060504,"[TITRE] ""Conducteur d'engins agricoles F-H"" (H...",A1101
